In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

# Standard library - import pathlib with explicit alias to avoid matplotlib.path conflict
import sys
import time
import re
import random
from pathlib import Path as PathLib

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import sklearn
from sklearn.cluster import KMeans, DBSCAN
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import umap
import anndata as ad
import scanpy as sc
from sknetwork.clustering import Louvain, Leiden
from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
import xgboost as xgb
import distinctipy
import networkx
from leidenalg import find_partition
import shap
import icecream as ic

# Add custom module paths


# Jupyter/IPython magic commands
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Plotting configuration
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

params = {
    'axes.titlesize': 30,
    'legend.fontsize': 20,
    'figure.figsize': (6, 5),
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'figure.titlesize': 30
}
plt.rcParams.update(params)
sns.set_style("white")

# Analysis configuration
Run = "Corrs"
hKWD = {'element': 'step', 'fill': False, 'stat': 'density'}
pKWD = {'dpi': 200, 'bbox_inches': 'tight'}

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
# CyTOF Helper Package
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/HelperPackage/")
%load_ext autoreload
%autoreload 2
import cytof_helper
from cytof_helper.utils import get_markers, gate_cells
from cytof_helper.normalization import normalize_markers_optimization
from cytof_helper.plotting import *
from cytof_helper.stats import *
from cytof_helper.interactive import *


# Histogram Plotting
Using `plot_histograms_multi_df` from `cytof_helper.plotting`.

# Load and initialize

In [ ]:
# Define data directory using pathlib
data_dir = PathLib("/Users/ronguy/Dropbox/CyTOF_Breast/PDX_Tam/Data/")

In [ ]:
# Use pathlib glob to get file list
FList = list(data_dir.glob("*.csv"))

In [ ]:
FList.sort()
FList

In [ ]:
FList=[FList[-1]]
FList

In [ ]:
DBs=[str(f).split("/")[-1].split(".")[0] for f in FList]

In [ ]:
DBs

In [ ]:
for F,DB in zip(FList,DBs):
    print(F)
    globals()[DB]=pd.read_csv(F)

In [ ]:
# Load mapping file using pathlib
mapping_file = PathLib("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx")
Rep=dict(pd.read_excel(mapping_file).iloc[:,:].values)

In [ ]:
Rep['H3K16ac']='H4K16ac'

In [ ]:
Rep['pH2A.x']='pH2A.X'

In [ ]:
for F,DB in zip(FList,DBs):
    globals()[DB]=pd.read_csv(F)
    print(f"{DB} {globals()[DB].shape[0]}")
    globals()[DB].rename(columns=Rep,inplace=True)


In [ ]:
N=list(PDX2_mod.columns)
N.sort()

In [ ]:
N

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=get_markers(N,marker_file='/Users/ronguy/Dropbox/Work/CyTOF/Markers_Names.xlsx')

In [ ]:
plot_histograms_multi_df([np.arcsinh(globals()[db]/5) for db in DBs],
                         NamesAll,df_names=DBs,hist_kwargs=hKWD,ncols=3
                        );

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
    sns.histplot(data=D,x='H4',**hKWD,color='magenta')
#    sns.histplot(data=D,x='H2A',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

## Gate on H3.3/H2A too low, but also remove outliers 99.99% from all

# Gating
Using `gate_cells` from `cytof_helper.utils`.

In [ ]:
for DB in DBs:
    globals()[DB]=gate_cells(globals()[DB], name=DB)

In [ ]:
for db in DBs:
    globals()[db][NamesAll]=np.arcsinh(globals()[db][NamesAll]/5)
    globals()[db]['Line']=db

In [ ]:
CAll=pd.concat([globals()[db] for db in DBs]).copy()

In [ ]:
UM=umap.UMAP(verbose=True,min_dist=0.04,n_neighbors=15)

In [ ]:
X_2d=UM.fit_transform(CAll[NamesAll].sample(15000))
X_2d=UM.transform(CAll[NamesAll])

In [ ]:
AD=ad.AnnData(CAll)
AD.obsm['X_umap']=X_2d

In [ ]:
sns.histplot(X_2d[:,0])

In [ ]:
sc.pl.umap(AD,color=['CD298','mMHC','mCD45'],cmap='magma_r')

In [ ]:
M=X_2d[:,0]>4

In [ ]:
CAll=CAll[M]

In [ ]:
for DB in DBs:
    M=CAll['Line']==DB
    globals()[DB]=(np.sinh(CAll[M][NamesAll])*5).copy()
    print(DB,globals()[DB].shape)

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
    sns.histplot(data=D,x='H4',**hKWD,color='magenta')
#    sns.histplot(data=D,x='H2A',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

# Normalize using new method on all intercellular markers

# Normalization
Using `normalize_markers_optimization` from `cytof_helper.normalization` package instead of defining it inline.

In [ ]:
NormMRK

In [ ]:
for DB in DBs:
    globals()[DB]=normalize_markers_optimization(globals()[DB],norm_columns=['H3','H3.3','H4'],norm_markers=NormMRK)

In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)

In [ ]:
MRK_All=NamesAll.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')
#MRK_All.remove('H2A')

EPC=EpiCols.copy()
Core=['H3','H3.3','H4']#,'H2A']
for C in Core:
    EPC.remove(C)

In [ ]:
NC=5000
aaaa=pd.concat([globals()[DB].sample(NC, replace=False, random_state=RANDOM_SEED) for DB in DBs]).copy()
                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Line']=DB

In [ ]:
for DB in DBs:
    globals()[DB].to_parquet(f"Data/{DB}.parquet")

In [ ]:
DBCLR=dict(zip(DBs,distinctipy.get_colors(len(DBs))))

In [ ]:
plot_histograms_multi_df([globals()[db] for db in DBs],
                         MRK_All,df_names=DBs,hist_kwargs=hKWD,ncols=4);

In [ ]:
CAll=pd.concat([globals()[db].sample(25000,replace=False,random_state=RANDOM_SEED) for db in DBs]).copy()

In [ ]:
UM=umap.UMAP(verbose=True,min_dist=0.04,n_neighbors=20,random_state=RANDOM_SEED)

In [ ]:
MRK=['BMI1',
 'CD24',
 'CD44',
 'CD49f',
 'E-cadherin',
 'ER',
 'EpCAM',
 'GATA3',
 'H2AK119ub',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me1',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H4K16ac',
 'H4K20me3',
 'KI67',
 'KRT5',
 'KRT8-18',
 'MBD',
 'Pan-KRT',
 'Vimentin',
 'aSMA',
 'pH2A.X',
 'pH3']

In [ ]:
X_2d=UM.fit_transform(CAll[MRK])
#X_2d=UM.transform(CAll[MRK])

In [ ]:
CAll.reset_index(inplace=True,drop=True)

In [ ]:
AD=ad.AnnData(CAll[MRK],obs=CAll[['Line']])
AD.obsm['X_umap']=X_2d

In [ ]:
sc.pl.umap(AD,color=MRK+['Line'],cmap='seismic',vcenter=0)

In [ ]:
AD=ad.AnnData(CAll[MRK],obs=CAll[['Line']])
AD.obsm['X_umap']=X_2d

In [ ]:
lllbl=InteractiveClusterLabeler(AD,subsample=10000)

In [ ]:
lllbl.show()

In [ ]:
AD.obs['Cl']=lllbl.full_predicted_labels
AD.obs['Cl']=AD.obs['Cl'].astype('category')

In [ ]:
sc.pl.umap(AD,color='Cl')

In [ ]:
pd.concat([AD.to_df(),AD.obs],axis=1).to_parquet("Data/PDX2.parquet")


In [ ]:
AD.obsm.to_df().to_parquet("Data/PDX2_UMAP.parquet")

In [ ]:
sc.pl.umap(AD,color=MRK+['Line','Cl'],cmap='seismic',vcenter=0,show=False);
plt.savefig('Plots/CyTOF3_PDX2_UMAP.png',dpi=200,bbox_inches='tight')

In [ ]:
marker_sets = {
    # Luminal/epithelial program: epithelial & ER axis up; mesenchymal/basal down
    "Epithelial_Luminal": {
        "up":   {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"},
        "down": {"KRT5", "Vimentin", "aSMA"}
    },

    # # Basal-like program: basal keratin/mesenchymal up; luminal/epithelial down
    # "Basal_like": {
    #     "up":   {"KRT5",  "Vimentin"},
    #     "down": {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"}
    # },

    # “Basal_Noa” (histone-flavor): active H3K4 marks up; repressive H3K9me2 down
    "Basal_Noa": {
        "up":   {"H3K4me1", "H3K4me3","H3K9me2"},
        "down": {"H4K20me3","H3K36me3"}
    },

    # # EMT programs: Vimentin/aSMA/CD44 up; E-cadherin down
    # "EMT": {
    #     "up":   {"Vimentin", "aSMA", "CD44"},
    #     "down": {"E-cadherin"}
    # },
    

    # Proliferation / cell-cycle & immediate-early signaling up
    "Proliferation": {
        "up":   {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},
        "down": set()  
    },
}

In [ ]:
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *

In [ ]:
Z, P, Zabs, Pabs, Zdir = sipsic_like_scores_v3(
    AD, marker_sets,
    n_perm=2048,                   # or even 16 for smoke test
    prefer_permutation=True,     # <- important
    perm_batch=512,              # keeps RAM flat
    normalize_set_weights="l2",
    use_sparse_W=False,
    progress=True               # progress bars can add overhead in some envs
)

In [ ]:
%matplotlib inline

In [ ]:
ADUS=ad.AnnData(obs=Z)
ADUS.obsm['X_umap']=AD.obsm['X_umap']
#    ADUS.obs['Class']=AD.obs['Class'].values
sc.pl.umap(ADUS,color=list(ADUS.obs.columns),cmap='seismic',show=False,vcenter=0,);
plt.show()

In [ ]:
ADUS.obs['Cl']=AD.obs['Cl']
ADUS.obs['Cl']=ADUS.obs['Cl'].astype('category')

In [ ]:
ADUS.uns['Cl_colors']=distinctipy.get_colors(len(np.unique(lllbl.full_predicted_labels)))

In [ ]:
sc.pl.umap(ADUS,color=list(ADUS.obs.columns),cmap='seismic',show=False,vcenter=0,);
plt.savefig('Plots/CyTOF3_PDX2_UMAP_Sig.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
ADUS.obs.groupby('Cl').count()

In [ ]:
ADUS.obs.to_parquet("Data/PDX2_Sigs.parquet")

In [ ]:
SigLine=pd.concat([ADUS.obs,AD.obs],axis=1).copy()
